# 04 — Model Feature Selection and Temporal Cross-Validation

This notebook keeps feature generation separate from model input selection, compares manually defined logistic-regression schemas with expanding folds confined to the existing outer training split, and then sends one manually chosen schema through the unchanged outer validation harness. Outer validation and the locked test are not accessed during inner CV.

In [ ]:
from dataclasses import replace
from datetime import date
from pathlib import Path

PROVIDER = "yfinance"
TICKERS = ("AAK.ST", "SAAB-B.ST", "VOLV-B.ST")
DATA_CUTOFF = date(2025, 12, 31)

TRAIN_START = date(2010, 1, 1)
TRAIN_END = date(2021, 12, 31)
VALIDATION_START = date(2022, 1, 1)
VALIDATION_END = date(2023, 12, 31)
TEST_START = date(2024, 1, 1)
TEST_END = date(2025, 12, 31)

## Define one generated feature set and two model schemas

`FeatureSetSpec` still controls feature computation. `ModelSpec.feature_columns` only controls which generated columns, and in which order, enter the estimator. `None` retains all generated features in canonical order.

In [ ]:
from swingtrader.data.features import DEFAULT_FEATURE_SET
from swingtrader.modeling.datasets import UniverseSpec, V2_PRIMARY_TASK, V2_TARGET_SET
from swingtrader.modeling.experiments import (
    ExperimentSpec,
    ModelSpec,
    TemporalCrossValidationSpec,
    TemporalSplitSpec,
    resolve_model_feature_columns,
)
from swingtrader.modeling.training import LOGISTIC_REGRESSION_MODEL_TYPE

feature_set = DEFAULT_FEATURE_SET.select(
    "returns",
    "trend",
    "momentum",
    "volatility",
    "volume",
    name="notebook_cv_candidates",
    version="1",
)

common_hyperparameters = {
    "regularization_strength": 1.0,
    "max_iter": 1_000,
    "tolerance": 1e-8,
}
all_feature_model = ModelSpec(
    name="logistic_all_features",
    version="1",
    model_type=LOGISTIC_REGRESSION_MODEL_TYPE,
    hyperparameters=common_hyperparameters,
    feature_columns=None,
)
selected_feature_model = ModelSpec(
    name="logistic_selected_features",
    version="1",
    model_type=LOGISTIC_REGRESSION_MODEL_TYPE,
    hyperparameters=common_hyperparameters,
    feature_columns=(
        "return_5d",
        "return_20d",
        "close_to_ema_fast",
        "adx",
        "rsi",
        "atr_percent",
        "turnover_zscore",
    ),
)

In [ ]:
universe = UniverseSpec(
    name="notebook_training_universe",
    version="1",
    provider=PROVIDER,
    tickers=TICKERS,
)
split_spec = TemporalSplitSpec(
    name="notebook_fixed_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)
experiment_spec = ExperimentSpec(
    name="notebook_feature_selection_cv",
    version="1",
    feature_set=feature_set,
    target_set=V2_TARGET_SET,
    task=V2_PRIMARY_TASK,
    universe=universe,
    data_cutoff=DATA_CUTOFF,
    split=split_spec,
    model=all_feature_model,
    random_seeds={"model": 42, "evaluation": 43},
)

## Build the canonical dataset and fixed outer split

Feature generation runs once for the declared feature set. The inner fold builder receives the resulting bundle and may only use positions already assigned to outer train.

In [ ]:
from swingtrader.data.db import resolve_database_engine
from swingtrader.modeling.datasets import build_temporal_dataset
from swingtrader.modeling.experiments import FixedTemporalSplitter

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
database_url = (
    f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
)
engine = resolve_database_engine(database_url=database_url)

bundle = build_temporal_dataset(engine=engine, spec=experiment_spec.dataset_spec)
split_result = FixedTemporalSplitter(experiment_spec.split).assign(bundle)

{
    "generated_feature_count": len(bundle.manifest.feature_columns),
    "outer_train": split_result.summary("train").to_manifest(),
    "outer_validation": split_result.summary("validation").to_manifest(),
}

## Inspect the resolved input order

The explicit tuple is preserved exactly. Unknown or duplicate names fail before fitting.

In [ ]:
{
    "all_features": resolve_model_feature_columns(
        all_feature_model, bundle.manifest.feature_columns
    ),
    "selected_features": resolve_model_feature_columns(
        selected_feature_model, bundle.manifest.feature_columns
    ),
}

## Compare manually defined candidates with train-only folds

Every fold uses global trading dates, keeps ticker rows for a date together, applies the stored target-resolution purge rule, and fits a fresh imputer, scaler, and estimator. The result intentionally contains only date boundaries, row counts, precision, recall, and ROC AUC.

In [ ]:
import pandas as pd

from swingtrader.modeling.training import (
    EvaluationConfig,
    run_baseline_cross_validation,
)

cv_spec = TemporalCrossValidationSpec(
    n_folds=4,
    validation_sessions=63,
    minimum_train_sessions=504,
)
evaluation_config = EvaluationConfig(random_seed=43)

cv_results = {}
for name, model_spec in {
    "all_features": all_feature_model,
    "selected_features": selected_feature_model,
}.items():
    candidate = replace(experiment_spec, name=f"notebook_cv_{name}", model=model_spec)
    cv_results[name] = run_baseline_cross_validation(
        bundle,
        split_result,
        candidate,
        cv_spec,
        evaluation_config=evaluation_config,
    )

pd.concat(cv_results, names=["candidate", "row"])

## Manually choose a schema and use the existing outer harness

No winner-selection policy is implemented. After inspecting the fold-level train/validation gaps, choose a candidate explicitly. The normal baseline harness then fits once on all outer training rows and evaluates outer validation with its existing full report.

In [ ]:
from swingtrader.modeling.training import run_baseline_experiment

chosen_model = selected_feature_model
chosen_experiment = replace(
    experiment_spec,
    name="notebook_selected_logistic",
    model=chosen_model,
)
outer_result = run_baseline_experiment(
    bundle,
    split_result,
    chosen_experiment,
    ranking_return_column="forward_return_5d",
    evaluation_config=evaluation_config,
    include_locked_test=False,
    artifact_directory=(
        repo_root / "artifacts" / "notebook_feature_selection_cv"
    ),
)

{
    "resolved_feature_columns": outer_result.model.feature_columns,
    "validation_metrics": dict(
        outer_result.reports["validation"].aggregate_metrics
    ),
}

## Locked-test boundary

This notebook deliberately leaves locked-test evaluation disabled. Freeze the selected feature schema, preprocessing, model hyperparameters, classification threshold, and ranking rule before using `include_locked_test=True` in a separate final experiment.